**FairSpeech - Notebook 03**

**Author:** Jaswanth Maradani

**Role:** Data & Statistics Lead

**Objective:** Create the FairSpeech segmentation strategy, accent clusters, and gold dataset for ASR processing.

Outputs:
*   Data cleaning rules
*   Accent clustering
*   Segment table
*   Bronze/Silver/Gold design
*   Dataset handoff for ASR processing

In [1]:
from datasets import load_dataset
import pandas as pd

edacc = load_dataset(
    "edinburghcstr/edacc"
)

df = pd.DataFrame(
    edacc["validation"]
)

print(df.shape)

README.md:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

data/validation-00000-of-00006-17ec18f4d(…): reconstructing file:   0%|          |  0.00B /  457MB            

data/validation-00000-of-00006-17ec18f4d(…): downloading bytes:           |  0.00B            

data/validation-00001-of-00006-6f3978e4f(…): reconstructing file:   0%|          |  0.00B /  478MB            

data/validation-00001-of-00006-6f3978e4f(…): downloading bytes:           |  0.00B            

...

data/test-00009-of-00010-648ce5002d12449(…): reconstructing file:   0%|          |  0.00B /  299MB            

data/test-00009-of-00010-648ce5002d12449(…): downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/9848 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9289 [00:00<?, ? examples/s]

(9848, 7)


Remove IGNORE segments

In [2]:
df_clean = df[
    ~df["text"].str.contains(
        "IGNORE_TIME_SEGMENT",
        na=False
    )
].copy()

print(df_clean.shape)

(9786, 7)


Cleaning Rule 1

Removed segments containing IGNORE_TIME_SEGMENT_IN_SCORING.

Reason: These segments are excluded from ASR evaluation and should not contribute to fairness metrics.

In [3]:
df_clean["word_count"] = (
    df_clean["text"]
    .apply(lambda x: len(str(x).split()))
)

In [4]:
df_clean = df_clean[
    df_clean["word_count"] >= 5
]

Cleaning Rule 2

Removed segments with fewer than 5 words.

Reason:

Very short utterances such as

"yeah" "okay" "right"

cannot generate meaningful grammar feedback.

In [5]:
accent_mapping = {
    "Southern British English":
        "Native",

    "Mainstream US English":
        "Native",

    "Irish English":
        "Native",

    "Scottish English":
        "Native",

    "Indian English":
        "South Asian",

    "Jamaican English":
        "Caribbean",

    "South African English":
        "African",

    "Italian":
        "European",

    "Spanish":
        "European",

    "Catalan":
        "European",

    "Bulgarian":
        "European",

    "German":
        "European",

    "French":
        "European",

    "Chinese":
        "East Asian",

    "Japanese":
        "East Asian",

    "Vietnamese":
        "East Asian",

    "Egyptian":
        "Middle Eastern"
}

In [6]:
df_clean["accent_cluster"] = (
    df_clean["accent"]
    .map(accent_mapping)
)

Creating Segment IDs

In [7]:
df_clean["segment_id"] = (
    "SEG_" +
    df_clean.index.astype(str)
)

Building Gold Dataset

In [8]:
gold_df = df_clean[
    [
        "segment_id",
        "speaker",
        "accent",
        "accent_cluster",
        "l1",
        "word_count",
        "text",
        "audio"
    ]
]

In [9]:
gold_df.to_csv(
    "gold_dataset.csv",
    index=False
)

In [11]:
gold_df_export = gold_df.copy()

gold_df_export = gold_df_export.drop(
    columns=["audio"]
)

gold_df_export.to_parquet(
    "gold_dataset.parquet",
    index=False
)

gold_df_export.to_csv(
    "gold_dataset.csv",
    index=False
)

print("Dataset exported successfully")

Dataset exported successfully


In [12]:
print(gold_df.columns)

Index(['segment_id', 'speaker', 'accent', 'accent_cluster', 'l1', 'word_count',
       'text', 'audio'],
      dtype='object')


In [17]:
print(gold_df_export.columns)

Index(['segment_id', 'speaker', 'accent', 'accent_cluster', 'l1', 'word_count',
       'text'],
      dtype='object')


In [16]:
print(gold_df.shape)

gold_df["accent_cluster"].value_counts()

(5871, 8)


,count
accent_cluster,
Native,2213
European,901
East Asian,541
Middle Eastern,283
South Asian,204
Caribbean,201
African,164


**Accent Cluster Findings**

The processed dataset contains 5,871 analysis-ready segments.

Native English speakers constitute the largest cluster (2,213 segments), while African English represents the smallest cluster (164 segments).

This imbalance may affect statistical uncertainty during fairness evaluation. Therefore, all future results will be reported with speaker-level confidence intervals and group sample sizes.

In [19]:
gold_df_export.groupby(
    "accent_cluster"
).agg(
    segments=("segment_id","count"),
    speakers=("speaker","nunique")
)

,segments,speakers
accent_cluster,,
African,164,2
Caribbean,201,2
East Asian,541,7
European,901,12
Middle Eastern,283,2
Native,2213,23
South Asian,204,2


In [20]:
cluster_summary = (
    gold_df_export
    .groupby("accent_cluster")
    .agg(
        segments=("segment_id","count"),
        speakers=("speaker","nunique")
    )
)

cluster_summary.to_csv(
    "cluster_summary.csv"
)

**Handoff Summary for ASR Processing**

Gold Dataset Records: 5,871

Accent Clusters:

*   Native
*   European
*   East Asian
*   Middle Eastern
*   South Asian
*   Caribbean
*   African

Cleaning Rules:

1.  Removed IGNORE_TIME_SEGMENT records.

1.   Removed utterances shorter than 5 words.

**Prepared By:** Jaswanth Maradani Data & Statistics Lead

**Purpose:** Dataset preparation for Whisper, CrisperWhisper and Vosk processing.